# ROGII: Honest 773-Well Baseline and Submission Audit

This notebook establishes a deliberately simple, leakage-safe reference point for the ROGII Wellbore Geology Prediction competition. Its purpose is not to claim a competitive method. It answers three practical questions before more complex GR alignment, particle filtering, HMM, or learned residual models are trusted:

1. What does the last visible TVT value score at the real prediction boundary across all 773 training wells?
2. How heavy is the per-well error tail behind the pooled row-wise RMSE?
3. Can we generate a submission whose IDs, ordering, row count, and finite values are checked against the official sample?

The test prediction for a well is the final non-null `TVT_input` repeated over its missing suffix. No hidden training target, train/test filename overlap, public submission artifact, or leaderboard-tuned constant is used. This makes the notebook useful as a floor, a debugging oracle, and a safe fallback for later per-well gating.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_CANDIDATES = [
    Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
    Path('/kaggle/input/rogii-wellbore-geology-prediction'),
    Path('data/raw/competition'),
    Path('../../data/raw/competition'),
]
DATA_ROOT = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Competition data directory was not found')
TRAIN_DIR = DATA_ROOT / 'train'
TEST_DIR = DATA_ROOT / 'test'
SAMPLE_PATH = DATA_ROOT / 'sample_submission.csv'
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Data root:', DATA_ROOT)
print('Training horizontal wells:', len(list(TRAIN_DIR.glob('*__horizontal_well.csv'))))
print('Visible test horizontal wells:', len(list(TEST_DIR.glob('*__horizontal_well.csv'))))

## Validation protocol

Each training CSV already contains the deployment-shaped boundary: `TVT_input` is known on a prefix and missing on a contiguous suffix, while `TVT` contains truth for local evaluation. We use that boundary exactly as supplied. Errors are accumulated across every suffix row before taking the square root, matching the official pooled RMSE. We also report macro, median, p90, p95, and worst per-well RMSE because a respectable pooled mean can hide catastrophic branch failures.

Random row splits are inappropriate here: adjacent rows from the same well are strongly dependent, and the operational task is future suffix prediction.

In [ ]:
def split_boundary(tvt_input: pd.Series) -> int:
    observed = tvt_input.notna().to_numpy()
    missing = np.flatnonzero(~observed)
    if len(missing) == 0 or missing[0] == 0:
        raise ValueError('Expected a visible prefix followed by a missing suffix')
    boundary = int(missing[0])
    if observed[boundary:].any():
        raise ValueError('Missing TVT_input values are not a contiguous suffix')
    return boundary

records = []
total_sse = 0.0
total_rows = 0
for path in sorted(TRAIN_DIR.glob('*__horizontal_well.csv')):
    frame = pd.read_csv(path, usecols=['TVT', 'TVT_input'])
    boundary = split_boundary(frame['TVT_input'])
    anchor = float(frame.loc[boundary - 1, 'TVT_input'])
    truth = frame.loc[boundary:, 'TVT'].to_numpy(dtype=float)
    error = anchor - truth
    sse = float(error @ error)
    n_eval = len(error)
    total_sse += sse
    total_rows += n_eval
    records.append({
        'well_id': path.name.removesuffix('__horizontal_well.csv'),
        'n_eval': n_eval,
        'rmse': math.sqrt(sse / n_eval),
    })

cv = pd.DataFrame(records)
summary = pd.Series({
    'wells': len(cv),
    'evaluation rows': total_rows,
    'pooled RMSE': math.sqrt(total_sse / total_rows),
    'macro well RMSE': cv.rmse.mean(),
    'median well RMSE': cv.rmse.median(),
    'p90 well RMSE': cv.rmse.quantile(0.90),
    'p95 well RMSE': cv.rmse.quantile(0.95),
    'worst well RMSE': cv.rmse.max(),
})
display(summary.to_frame('value').style.format('{:.4f}'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(cv.rmse, bins=35, color='#2a6fbb', alpha=0.85)
axes[0].axvline(cv.rmse.quantile(0.95), color='#c23b22', linestyle='--', label='p95')
axes[0].set(title='Per-well RMSE distribution', xlabel='RMSE', ylabel='Wells')
axes[0].legend()
axes[1].scatter(cv.n_eval, cv.rmse, s=11, alpha=0.45, color='#347a4a')
axes[1].set(title='Tail risk versus suffix length', xlabel='Evaluation rows', ylabel='Well RMSE')
plt.tight_layout()
plt.show()
display(cv.nlargest(10, 'rmse'))

## What this baseline teaches us

The last-value predictor is surprisingly hard to beat blindly because many target paths remain close to the final visible TVT. However, its upper tail is severe. A useful advanced method therefore needs more than a slightly better average well: it must reduce large branch errors without creating new catastrophic wells.

For future experiments I recommend retaining this exact table and comparing candidates on pooled RMSE, p95/worst-well error, and per-well win/loss. Candidate selection or overrides must use features observable in the current well's prefix; using suffix truth, duplicated training targets, or leaderboard feedback as a per-well selector would invalidate the estimate.

## Build and audit `submission.csv`

Predictions are first stored by the official row ID (`well_id` plus original row index), then joined back to `sample_submission.csv`. This is safer than assuming filesystem order or concatenation order. The final assertions check the exact header, exact ID order, uniqueness, row count, and finite values.

In [ ]:
sample = pd.read_csv(SAMPLE_PATH)
assert list(sample.columns) == ['id', 'tvt']
prediction_by_id = {}
for path in sorted(TEST_DIR.glob('*__horizontal_well.csv')):
    well_id = path.name.removesuffix('__horizontal_well.csv')
    frame = pd.read_csv(path, usecols=['TVT_input'])
    boundary = split_boundary(frame['TVT_input'])
    anchor = float(frame.loc[boundary - 1, 'TVT_input'])
    for row_index in range(boundary, len(frame)):
        prediction_by_id[f'{well_id}_{row_index}'] = anchor

sample_ids = sample['id'].astype(str)
assert set(prediction_by_id) == set(sample_ids), 'Prediction IDs do not match sample IDs'
submission = pd.DataFrame({
    'id': sample_ids,
    'tvt': sample_ids.map(prediction_by_id).astype(float),
})
assert list(submission.columns) == ['id', 'tvt']
assert submission['id'].tolist() == sample_ids.tolist()
assert submission['id'].is_unique
assert len(submission) == len(sample)
assert np.isfinite(submission['tvt']).all()
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
print('Saved:', OUTPUT_DIR / 'submission.csv')
print('Rows:', len(submission))
print('TVT range:', float(submission.tvt.min()), float(submission.tvt.max()))
display(submission.head())

## Reproducibility note

This notebook is deterministic, CPU-only, internet-off, and has no external datasets or model artifacts. The validation number is a training-data estimate, not a promise about the public or private leaderboard. Its intended value is methodological: establish a trustworthy floor, preserve row-level submission integrity, and require every complex tracker to demonstrate independent out-of-well value before deployment.